<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 2: Case B and Constraint Structure

**The advertising plan is acceptable only when both individual spending limits and the shared budget requirement hold.**

Part 1 introduced the standard symbols through the unconstrained clock case. This part formulates the advertising case from 01-2 and uses it to distinguish bound-constrained and generally constrained problems.

### 1 · Carry forward the advertising decision

The club chooses social-media spending \(s\) and flyer spending \(r\), both in dollars. The forecast reach is \(R(s,r)=4s+2r\). The requirements are

> $\displaystyle s+r\le100,\qquad 0\le s\le70,\qquad 10\le r\le100.$

The spending changes the real advertising plan. Forecast reach and total spending are produced quantities.

### 2 · Translate every requirement into the standard form

Let

> $\displaystyle x=\begin{bmatrix}s\\r\end{bmatrix},\qquad
y=\operatorname{Sim}(x)=\begin{bmatrix}R(x)\\B(x)\end{bmatrix}
=\begin{bmatrix}4s+2r\\s+r\end{bmatrix}.$

Because the shared template minimizes a scalar objective, maximizing reach is written as minimizing its negative:

> $\displaystyle f(y)=-R(x).$

Write every inequality as a residual that must be nonpositive:

| Requirement | Residual form |
|:---|:---|
| \(s\ge0\) | \(g_1(x)=-s\le0\) |
| \(s\le70\) | \(g_2(x)=s-70\le0\) |
| \(r\ge10\) | \(g_3(x)=10-r\le0\) |
| \(r\le100\) | \(g_4(x)=r-100\le0\) |
| \(s+r\le100\) | \(g_5(x)=s+r-100\le0\) |

There are \(m_g=5\) inequality constraints and \(m_h=0\) equality constraints. A negative residual has slack, zero is active, and a positive residual is a violation.

The complete formulation is

> $\displaystyle \underset{x\in\mathbb R^2}{\operatorname{minimize}}\quad -(4s+2r)$
>
> $\displaystyle \text{subject to}\quad g_j(x)\le0\quad(j=1,\ldots,5).$

### 3 · Evaluate feasibility before reach

The code keeps response calculation, objective calculation, and constraint checks separate. The search uses a 1-dollar grid only to make the candidate set visible.

In [ ]:
import numpy as np


def simulate_advertising(x):
    social, flyers = np.asarray(x, dtype=float)
    return {"reach": 4.0 * social + 2.0 * flyers, "spending": social + flyers}


def objective_function(y):
    return -float(y["reach"])


def inequality_constraints(x, y):
    social, flyers = np.asarray(x, dtype=float)
    return np.array([
        -social,
        social - 70.0,
        10.0 - flyers,
        flyers - 100.0,
        y["spending"] - 100.0,
    ])


def evaluate_candidate(x):
    decision = np.asarray(x, dtype=float)
    response = simulate_advertising(decision)
    residuals = inequality_constraints(decision, response)
    return {
        "x": decision,
        "y": response,
        "f": objective_function(response),
        "g": residuals,
        "feasible": bool(np.all(residuals <= 1e-10)),
    }

In [ ]:
for decision in ([70, 30], [70, 40]):
    record = evaluate_candidate(decision)
    print(
        f"x=({record['x'][0]:.0f}, {record['x'][1]:.0f}): "
        f"reach={record['y']['reach']:.0f}, "
        f"max g={record['g'].max():.0f}, feasible={record['feasible']}"
    )

levels = np.arange(0.0, 101.0, 1.0)
grid_records = [
    evaluate_candidate([social, flyers])
    for social in levels
    for flyers in levels
]
best_grid = min(
    (record for record in grid_records if record["feasible"]),
    key=lambda record: record["f"],
)
print(
    "Best 1-dollar grid candidate: "
    f"x=({best_grid['x'][0]:.0f}, {best_grid['x'][1]:.0f}), "
    f"reach={best_grid['y']['reach']:.0f}"
)

The plan \((70,40)\) has greater reach than \((70,30)\), but its shared-budget residual is \(10>0\). It is rejected before objective values are compared. The reported result is the best candidate on the stated 1-dollar grid.

### 4 · Change only the restrictions

Keep the same decision and objective, then compare three formulations:

| Formulation | Restrictions retained | Classification |
|:---|:---|:---|
| A | None; \(x\in\mathbb R^2\) | Unconstrained |
| B | Only \(0\le s\le70\) and \(10\le r\le100\) | Bound-constrained |
| C | Bounds and \(s+r\le100\) | Generally constrained |

Formulation A is unbounded: forecast reach can increase without limit. “Unconstrained” names the restriction structure; it does not guarantee that a useful optimum exists.

Formulation C is generally constrained because \(s+r\le100\) couples the two decisions. An exact requirement would use an equality residual. For example, spending exactly \$100 is \(h_1(x)=s+r-100=0\).

### 5 · Classify the full Case B formulation

Case B is a **generally constrained, continuous, single-objective, linear, direct algebraic, deterministic optimization problem**. Both the objective and every constraint are linear in \(x\).

### Takeaway

Convert requirements before comparing candidates:

> **write bounds and residuals → check every \(g_j\le0\) and \(h_k=0\) → reject any violation → compare \(f\) only among feasible candidates**

Removing or adding a requirement changes the optimization formulation even when the real decision and response calculation stay the same. Part 3 changes the allowed value types instead.